In [ ]:
# Create the total data
Total_Data <- setDT(readRDS(file = "Data/Q_Total_Data84.rds"))
Total_Data <- melt(
  Total_Data,
  measure.vars = names(Total_Data)[
    which(names(Total_Data) == "A2ME"):which(names(Total_Data) == "VAR")
  ],
  variable.name = "Char",    # equivalent to pivot_longer names_to
  value.name = "Rank",        # equivalent to pivot_longer values_to
  variable.factor = F
)
setorder(Total_Data, date, permno, Char)  # sort in place

# Create the total date tibble with 6 months lag to predict next 3 month return (3 quarters in total)
Date_table <- data.table(date = unique(Total_Data$date))[, `:=` (UseDate = shift(date, n = 3, fill=NA, type = "lead"))]
Date_table[, `:=` (Year = UseDate%/%10000,
                   Quarter = UseDate%%10000%/%100/3)]


Total_Data[, setdiff(names(Total_Data), c("Char", "permno", "date", "Early_Years", "Rank")) := NULL]

gc()


Char_data <- Total_Data |>
  dcast(
    ... ~ Char,                               # Fill in appropriate LHS (ID variables)
    value.var = c("Rank")
  )
gc()
Char_data <- merge(Char_data, Date_table, by = c("date"), all.x = T, sort = F)
Char_data <- merge(setDT(readRDS("Data/Macro_vars.rds")), Char_data, by = c("Year", "Quarter"), all = FALSE, sort = F)
Char_data[,c("Quarter", "date") := NULL]
rm(Total_Data)
gc()

# Make SIC dummy
SICs <- sort(unique(Char_data$SIC))

# omit first category as baseline
Char_data[, paste0("SIC_", SICs[-1]) :=
            lapply(SICs[-1], function(x)
              as.integer(SIC == x))]

Char_data[is.na(Char_data)] <- 0

# Add the char, interaction, industry code
train_matrix <- as.matrix(Char_data[, Rank_A2ME:Rank_VAR])
train_matrix <- cbind(train_matrix,
                      kronecker(train_matrix, matrix(1, 1, 8)) * kronecker(matrix(1, 1, ncol(train_matrix)), as.matrix(Char_data[, dp:svar])),
                      as.matrix(Char_data[, SIC_1:SIC_99]))